# Volume 4, Chapter 10: The zeta function (Python companion)

Exercises first, solutions at the end. We watch the Euler product converge, compute $\zeta$ beyond its series with Riemann's own formula, see $-\frac1{12}$ emerge from smoothed sums, implement the Riemann–Siegel formula, find and count zeros on the critical line, and sieve the Möbius function. Requirements: numpy and mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P10.1 (★) The Euler product and coprime pairs.** Sieve the primes up to $10^6$ and compare $\prod_{p\le P}(1-p^{-s})^{-1}$ with $\zeta(s)$ for $s=2,3$ and several $P$ (Theorem 10.3). How does the error depend on $P$? Then compute exactly the fraction of coprime pairs in $\{1,\dots,N\}^2$ with a Möbius sieve, and compare with $\frac6{\pi^2}$ (Corollary 10.4).

**P10.2 (★★) Beyond the series.** Implement Riemann's formula of Theorem 10.8 and compute $\zeta(s)$ with it at $s=2$, $\frac12$, $\frac12+14.1347\ldots i$, $-1$, $-3$ and $-2.5+3i$. Compare with `mpmath.zeta` and with Theorem 10.10. Then compute $\zeta(\frac12)$ a second way, from the alternating series of Exercise 10.3, accelerated by repeatedly averaging consecutive partial sums.

**P10.3 (★) Smoothed divergent sums.** Compute $\sum_{n\ge1}n^ke^{-n/N}-k!\,N^{k+1}$ for $k=0,\dots,4$ and $N=10,100,1000$, and compare with $\zeta(-k)$ (Proposition 10.11). Then replace $e^{-x}$ by other cut-offs $\psi$ with $\psi(0)=1$, such as $e^{-x^2}$ and $\frac1{\cosh x}$: what changes, and what does not?

**P10.4 (★★) CS application: the Riemann–Siegel formula.** Computing $\zeta(\frac12+it)$ from its definition is hopeless. The Riemann–Siegel formula, found in Riemann's unpublished notes by Siegel in 1932, gives Hardy's $Z$ (Figure 10.2) with only $N=\lfloor\sqrt{t/2\pi}\rfloor$ terms:
$$Z(t)\approx2\sum_{n\le N}\frac{\cos(\vartheta(t)-t\log n)}{\sqrt n}+(-1)^{N-1}\Bigl(\frac t{2\pi}\Bigr)^{-1/4}\frac{\cos2\pi(p^2-p-\frac1{16})}{\cos2\pi p},$$
where $p=\sqrt{t/2\pi}-N$ and $\vartheta(t)\approx\frac t2\log\frac t{2\pi}-\frac t2-\frac\pi8+\frac1{48t}+\frac7{5760t^3}$ (Stirling's series for $\log\Gamma$, Chapter 11). Implement it and compare with `mpmath.siegelz` for $t=20,100,1000,10000$.

**P10.5 (★★) Finding and counting zeros.** Locate the zeros of $Z$ in $[10,T]$ by sign changes and bisection. Compare the first ten with `mpmath.zetazero`, and count the zeros up to $T=100,500,1000$. Compare the counts with the Riemann–von Mangoldt formula $N(T)\approx\frac T{2\pi}\log\frac T{2\pi e}+\frac78$, which Volume 5 proves with the argument principle (Theorem 9.37).

**P10.6 (★) The Mertens function.** Sieve $\mu(n)$ for $n\le10^6$ and compute $M(x)=\sum_{n\le x}\mu(n)$, the summatory function of the coefficients of $\frac1\zeta$. Compare $|M(x)|$ with $\sqrt x$.

## Solutions

### P10.1

In [ ]:
import numpy as np
import mpmath as mp
from math import gcd
mp.mp.dps = 30
def primes_upto(n):
    sieve = np.ones(n + 1, dtype=bool); sieve[:2] = False
    for p in range(2, int(n**0.5) + 1):
        if sieve[p]:
            sieve[p*p::p] = False
    return np.nonzero(sieve)[0]
P = primes_upto(10**6)
for s in (2, 3):
    for bound in (10, 100, 10**4, 10**6):
        prod = mp.mpf(1)
        for p in P[P <= bound]:
            prod /= (1 - mp.mpf(int(p)) ** (-s))
        print(f"s = {s}: product over p <= {bound:>7}: {mp.nstr(prod, 15)}   error {mp.nstr(mp.zeta(s) - prod, 3)}")

In [ ]:
def coprime_fraction(N):
    mu = mobius_upto(N)
    return sum(int(mu[d]) * (N // d) ** 2 for d in range(1, N + 1)) / N**2
def mobius_upto(n):
    """Linear sieve for the Moebius function."""
    mu = np.ones(n + 1, dtype=np.int64); mu[0] = 0
    is_comp = np.zeros(n + 1, dtype=bool)
    for p in range(2, n + 1):
        if not is_comp[p]:
            mu[p::p] *= -1
            mu[p*p::p*p] = 0
            is_comp[2*p::p] = True
    return mu
for N in (10, 100, 1000, 10**4, 10**5):
    print(f"N = {N:>6}: fraction of coprime pairs = {coprime_fraction(N):.6f}")
print("6/pi^2 =", 6 / np.pi**2)

The error of the truncated product is about $\sum_{p>P}p^{-s}$, roughly $\frac{P^{1-s}}{(s-1)\log P}$: for $s=2$ it shrinks like $\frac1P$, for $s=3$ like $\frac1{P^2}$. The Euler product converges, but slowly; it is a tool for theory (zeros, primes), not for computing $\zeta$. The fraction of coprime pairs settles at $0.6079\ldots=\frac6{\pi^2}$, with error of order $\frac{\log N}N$. The sieve computes $\mu$ for all $n\le N$ at once: multiply by $-1$ along the multiples of each prime, and set to $0$ the multiples of each $p^2$.

### P10.2

In [ ]:
def zeta_theta(s):
    """Riemann's formula: Lambda(s) = -1/s - 1/(1-s) + int_1^oo omega(t) (t^{s/2} + t^{(1-s)/2}) dt/t."""
    omega = lambda t: mp.nsum(lambda n: mp.exp(-mp.pi * n * n * t), [1, mp.inf])
    I = mp.quad(lambda t: omega(t) * (t ** (s / 2) + t ** ((1 - s) / 2)) / t, [1, 3, mp.inf])
    Lam = -1 / s - 1 / (1 - s) + I
    return Lam * mp.pi ** (s / 2) / mp.gamma(s / 2)
def zeta_eta(s, terms=60):
    """zeta(s) = eta(s)/(1 - 2^(1-s)), eta summed with Euler-type averaging of partial sums (convergent for Re s > 0)."""
    partial = [mp.mpf(0)]
    for n in range(1, terms + 1):
        partial.append(partial[-1] + (-1) ** (n - 1) * mp.mpf(n) ** (-s))
    S = partial[1:]
    for _ in range(terms // 2):                       # repeated averaging of consecutive partial sums
        S = [(a + b) / 2 for a, b in zip(S, S[1:])]
    return S[-1] / (1 - 2 ** (1 - s))
for s in (mp.mpf(2), mp.mpf("0.5"), mp.mpc("0.5", "14.134725141734693790"), mp.mpf(-1), mp.mpf(-3), mp.mpc(-2.5, 3)):
    print(f"s = {mp.nstr(s, 8):>22}: theta formula {mp.nstr(zeta_theta(s), 15):>34}   mpmath zeta {mp.nstr(mp.zeta(s), 15)}")
print("zeta(1/2) by the eta series with averaging:", mp.nstr(zeta_eta(mp.mpf("0.5")), 15))
print("-B_2/2 =", -mp.bernoulli(2) / 2, "  -B_4/4 =", -mp.bernoulli(4) / 4)

Riemann's formula reproduces $\zeta$ everywhere, including at the first zero, where both computations give a value of order $10^{-19}$ (the zero is known to $20$ digits only in our input), at the negative integers $-\frac1{12}=-\frac{B_2}2$ and $\frac1{120}=-\frac{B_4}4$, and at a point with negative real part where no series converges. The integral over $[1,\infty[$ converges extremely fast because $\omega(t)\approx e^{-\pi t}$: this is the theta transformation doing the work, as in Chapter 8. The alternating series converges for $\operatorname{Re}s>0$ but slowly; averaging consecutive partial sums (Euler's transform in disguise, met in the Chapter 3 notebook) brings $60$ terms to $15$ digits.

### P10.3

In [ ]:
print("        N = 10            N = 100           N = 1000          zeta(-k)")
for k in range(0, 5):
    row = []
    for N in (10, 100, 1000):
        S = mp.nsum(lambda n: n**k * mp.exp(-n / mp.mpf(N)), [1, mp.inf])
        row.append(S - mp.factorial(k) * mp.mpf(N) ** (k + 1))
    print(f"k = {k}: " + "  ".join(f"{mp.nstr(v, 10):>16}" for v in row) + f"  {mp.nstr(mp.zeta(-k), 10):>16}")
N = 1000
for name, psi in (("e^(-x)", lambda x: mp.exp(-x)), ("e^(-x^2)", lambda x: mp.exp(-x * x)), ("1/cosh(x)", lambda x: 1 / mp.cosh(x))):
    S = mp.fsum(n * psi(mp.mpf(n) / N) for n in range(1, 60 * N))
    c = mp.quad(lambda x: x * psi(x), [0, mp.inf])
    print(f"cut-off {name:10s}: N^2 * int x psi = {mp.nstr(N**2 * c, 12):>16}   constant term = {mp.nstr(S - N**2 * c, 10)}")

For every $k$ the constant term is $\zeta(-k)$: $-\frac12$, $-\frac1{12}$, $0$, $\frac1{120}$, $0$, with errors of order $N^{-1}$ or $N^{-2}$. With other cut-offs, the divergent part $N^2\int_0^\infty x\psi(x)\,dx$ changes ($10^6$, $\frac12\cdot10^6$, about $1.83\cdot10^6$), but the constant term is $-\frac1{12}$ every time. This cut-off independence is the precise meaning of "$1+2+3+\cdots=-\frac1{12}$". The Euler–Maclaurin formula of Chapter 11 proves it: for $f(x)=x\psi(x/N)$, the sum is $\int_0^\infty f$ minus $\frac{B_2}2f'(0)=\frac1{12}\psi(0)$, plus terms that vanish as $N\to\infty$.

### P10.4

In [ ]:
from math import pi, log, cos, sqrt, floor
def theta_rs(t):
    return t / 2 * log(t / (2 * pi)) - t / 2 - pi / 8 + 1 / (48 * t) + 7 / (5760 * t**3)
def Z_rs(t):
    """Riemann-Siegel formula with the first correction term."""
    a = sqrt(t / (2 * pi)); N = floor(a); p = a - N
    th = theta_rs(t)
    main = 2 * sum(cos(th - t * log(n)) / sqrt(n) for n in range(1, N + 1))
    psi = cos(2 * pi * (p * p - p - 1 / 16)) / cos(2 * pi * p)
    return main + (-1) ** (N - 1) * a ** -0.5 * psi
for t in (20.0, 100.0, 1000.0, 10000.0):
    print(f"t = {t:7.0f}: Riemann-Siegel Z = {Z_rs(t):+.10f}   mpmath siegelz = {float(mp.siegelz(t)):+.10f}   terms used: {floor(sqrt(t/(2*pi)))}")

With one term the formula is already right to $3\cdot10^{-3}$ at $t=20$; at $t=10^4$, $39$ terms give $Z$ to $5\cdot10^{-6}$. The error of the formula with one correction term is $O(t^{-3/4})$, and further correction terms improve it. The cost is $O(\sqrt t)$ operations per value instead of the $O(t)$ terms needed by Euler–Maclaurin. This is how zeros are computed at large heights; the Odlyzko–Schönhage algorithm then evaluates many values at once with the FFT of Chapter 7.

### P10.5

In [ ]:
def zeros_in(T0, T1, step=0.05):
    ts = np.arange(T0, T1, step)
    vals = [Z_rs(t) for t in ts]
    out = []
    for a, b, fa, fb in zip(ts, ts[1:], vals, vals[1:]):
        if fa * fb < 0:
            lo, hi = a, b
            for _ in range(50):
                mid = (lo + hi) / 2
                if Z_rs(lo) * Z_rs(mid) <= 0: hi = mid
                else: lo = mid
            out.append((lo + hi) / 2)
    return out
zs = zeros_in(10, 100)
print("first ten zeros (Riemann-Siegel):", [round(z, 6) for z in zs[:10]])
print("mpmath zetazero               :", [round(float(mp.im(mp.zetazero(n))), 6) for n in range(1, 11)])
for T in (100, 500, 1000):
    count = len(zeros_in(10, T, 0.02))
    rvm = T / (2 * pi) * log(T / (2 * pi * np.e)) + 7 / 8
    print(f"T = {T:5d}: sign changes found = {count:4d}   mpmath nzeros = {mp.nzeros(T):4d}   (T/2pi) log(T/2pi e) + 7/8 = {rvm:7.1f}")

The first ten zeros agree with mpmath to about $10^{-2}$ at $t=14$ and to $10^{-3}$ at $t=50$: the Riemann–Siegel formula is an asymptotic expansion, less accurate at small heights. The counts $29$, $269$, $649$ agree with mpmath and with the Riemann–von Mangoldt formula to within $1$. That every zero up to $T$ is a sign change of $Z$ (so lies on the critical line) is confirmed here because the number of sign changes equals the total number of zeros in the critical strip, which mpmath computes independently. Turing's method for verifying the Riemann hypothesis up to a height rests on this comparison.

### P10.6

In [ ]:
M = np.cumsum(mobius_upto(10**6))
xs = np.arange(1, 10**6 + 1)
ratio = np.abs(M[1:]) / np.sqrt(xs)
r = ratio[1:]                                   # x >= 2
print("max |M(x)|/sqrt(x) for 2 <= x <= 10^6:", r.max().round(4), "at x =", int(xs[1:][r.argmax()]))
r2 = ratio[999:]                                # x >= 1000
print("max |M(x)|/sqrt(x) for 10^3 <= x <= 10^6:", r2.max().round(4), "at x =", int(xs[999:][r2.argmax()]))
for x in (10, 100, 1000, 10**4, 10**5, 10**6):
    print(f"M({x}) = {int(M[x]):5d}")

$|M(x)|$ stays below $\sqrt x$ in this range, and far below it for large $x$. Mertens conjectured in 1897 that $|M(x)|<\sqrt x$ for all $x>1$; Odlyzko and te Riele disproved it in 1985, without exhibiting a counterexample: there is none below $10^{16}$, and the first one is astronomically large. The weaker statement $M(x)=O(x^{1/2+\varepsilon})$ for every $\varepsilon>0$ is equivalent to the Riemann hypothesis: through $\frac1{\zeta(s)}=\sum\mu(n)n^{-s}$, the growth of $M$ controls how far to the left $\frac1\zeta$ is holomorphic, that is, where the zeros of $\zeta$ can be.